# 21. Protein Structure in Three Dimensions

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `tour.py`

What is inside a structure file: normal and sickle deoxyhemoglobin.


In [ ]:
# What is inside a structure file: normal and sickle deoxyhemoglobin.
from Bio.PDB import MMCIFParser

parser = MMCIFParser(QUIET=True)
for code in ("2HHB", "2HBS"):
    model = parser.get_structure(code, f"../data/{code}.cif")[0]
    print(f"{code}: {len(model)} chains")
    for chain in model:
        residues = [r for r in chain if r.id[0] == " "]  # amino acids
        others = sorted(
            {r.get_resname() for r in chain if r.id[0] != " "}
        )
        kind = "beta" if len(residues) == 146 else "alpha"
        line = f"  {chain.id}  {kind:5} {len(residues)} residues"
        if kind == "beta":
            line += f", residue 6 is {chain[6].get_resname()}"
        print(f"{line}; also {', '.join(others)}")

## `heme.py`

How the heme's iron sits between the two histidines in each beta chain of normal deoxyhemoglobin.


In [ ]:
# How the heme's iron sits between the two histidines in each beta chain
# of normal deoxyhemoglobin.
from Bio.PDB import MMCIFParser

model = MMCIFParser(QUIET=True).get_structure("2HHB", "../data/2HHB.cif")[
    0
]
for chain_id in ("B", "D"):
    chain = model[chain_id]
    heme = next(r for r in chain if r.get_resname() == "HEM")
    iron = heme["FE"]
    proximal = (
        iron - chain[92]["NE2"]
    )  # subtracting atoms gives angstroms
    distal = iron - chain[63]["NE2"]
    print(
        f"chain {chain_id}: iron to His92 {proximal:.2f} A, "
        f"to His63 {distal:.2f} A"
    )

## `surface.py`

How much of each key residue is exposed on the molecule's surface.


In [ ]:
# How much of each key residue is exposed on the molecule's surface.
from Bio.PDB import MMCIFParser
from Bio.PDB.DSSP import residue_max_acc
from Bio.PDB.SASA import ShrakeRupley

LARGEST = residue_max_acc[
    "Sander"
]  # each amino acid's area fully exposed
RESIDUES = (6, 42, 63, 85, 88, 92)

for code in ("2HHB", "2HBS"):
    model = MMCIFParser(QUIET=True).get_structure(
        code, f"../data/{code}.cif"
    )[0]
    for chain in list(model):  # keep one hemoglobin molecule, A to D
        if chain.id not in "ABCD":
            model.detach_child(chain.id)
    ShrakeRupley().compute(model, level="R")
    print(code)
    for number in RESIDUES:
        residue = model["B"][number]
        share = residue.sasa / LARGEST[residue.get_resname()]
        print(
            f"  {residue.get_resname()} {number:3}  {residue.sasa:5.0f}"
            f" square angstroms, {share:4.0%} exposed"
        )

## `contacts.py`

In the sickle hemoglobin crystal, which residues of another hemoglobin molecule touch valine 6?


In [ ]:
# In the sickle hemoglobin crystal, which residues of another hemoglobin
# molecule touch valine 6?
from Bio.PDB import MMCIFParser, NeighborSearch

model = MMCIFParser(QUIET=True).get_structure("2HBS", "../data/2HBS.cif")[
    0
]
molecule = {chain: 1 if chain in "ABCD" else 2 for chain in "ABCDEFGH"}
search = NeighborSearch(list(model.get_atoms()))

for chain_id in ("B", "D", "F", "H"):
    valine = model[chain_id][6]
    closest = {}
    for atom in valine:
        if atom.get_id() in ("N", "CA", "C", "O"):  # side chain only
            continue
        for near in search.search(atom.coord, 4.5):
            residue = near.get_parent()
            chain = residue.get_parent().id
            if (
                molecule[chain] == molecule[chain_id]
                or residue.id[0] != " "
            ):
                continue
            key = (chain, residue.get_resname(), residue.id[1])
            closest[key] = min(closest.get(key, 99), atom - near)
    print(f"valine 6 of chain {chain_id}: {len(closest)} contacts")
    for (chain, name, number), distance in sorted(
        closest.items(), key=lambda item: item[0][2]
    ):
        print(f"  chain {chain} {name} {number:3}  {distance:.1f} A")

## `alphafold.py`

How close is AlphaFold's prediction of beta globin to the crystal?


In [ ]:
# How close is AlphaFold's prediction of beta globin to the crystal?
from Bio.PDB import MMCIFParser, PDBParser, Superimposer

crystal = MMCIFParser(QUIET=True).get_structure(
    "2HHB", "../data/2HHB.cif"
)[0]["B"]
predicted = PDBParser(QUIET=True).get_structure(
    "AF", "../data/AF-P68871-F1.pdb"
)[0]["A"]

# AlphaFold stores its confidence, pLDDT, in the B-factor column
confidence = [residue["CA"].bfactor for residue in predicted]
low = [n for n, c in enumerate(confidence, start=1) if c < 90]
print(
    f"mean pLDDT {sum(confidence) / len(confidence):.1f}, "
    f"below 90 at residues {low}"
)

# The prediction keeps the first methionine, so its numbers are one higher
fixed = [crystal[n]["CA"] for n in range(1, 147)]
moving = [predicted[n + 1]["CA"] for n in range(1, 147)]
fit = Superimposer()
fit.set_atoms(fixed, moving)
print(f"after superposition, {len(fixed)} alpha carbons differ")
print(f"  by {fit.rms:.2f} A on average (root mean square)")